# AI Challenge - TRAKE Submission Pipeline
Notebook này được tối ưu hóa cho Google Colab (đề xuất dùng GPU T4/L4).
Nhiệm vụ: Tải dữ liệu từ Google Drive, trích xuất Keyframes, mã hóa bằng SigLIP + E5, tìm kiếm và sinh ra file `submission.jsonl`.

In [ ]:
!pip install -q qdrant-client fastembed torch transformers huggingface_hub pyscenedetect gdown opencv-python pillow

In [ ]:
import os
import gdown

print("Đang tải Dữ liệu và Tasks từ Google Drive...")
# Tải Data (Videos)
data_id = '1SeMMrpEECpXvo3hE5q9WTl-yYrHt0GRd'
gdown.download(f'https://drive.google.com/uc?id={data_id}', 'video_corpus.zip', quiet=False)

# Tải Tasks
task_id = '1sjslvmbv_jP9O83rKqB4TzJ3YWSMPzdM'
gdown.download(f'https://drive.google.com/uc?id={task_id}', 'tasks.zip', quiet=False)

!unzip -q -o video_corpus.zip -d /content/data/
!unzip -q -o tasks.zip -d /content/tasks/

In [ ]:
import json
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel
from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import glob
import uuid

# 1. KHỞI TẠO MODEL
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading SigLIP...")
vision_model_name = "google/siglip-so400m-patch14-384"
vision_processor = AutoProcessor.from_pretrained(vision_model_name)
vision_model = AutoModel.from_pretrained(vision_model_name).to(device)
vision_model.eval()

print("Loading Multilingual E5...")
text_model = TextEmbedding(model_name="intfloat/multilingual-e5-large")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="aic_collection",
    vectors_config={
        "image_siglip": VectorParams(size=1152, distance=Distance.COSINE),
        "text_e5": VectorParams(size=1024, distance=Distance.COSINE),
    }
)

def encode_image(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = vision_processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = vision_model.get_image_features(**inputs)
            if hasattr(image_features, 'pooler_output'):
                image_features = image_features.pooler_output
            elif hasattr(image_features, 'image_embeds'):
                image_features = image_features.image_embeds
            image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        return image_features.squeeze(0).cpu().tolist()
    except Exception as e:
        print(f"Lỗi mã hóa ảnh: {e}")
        return None

def encode_text(text):
    return list(text_model.embed([text]))[0].tolist()


In [ ]:
# 2. INGESTION (Xử lý Video & Nạp Qdrant)
# Đơn giản hóa: Chụp frame ở giữa video (Do yêu cầu tài nguyên lớn, đoạn này giả lập logic cắt frame cơ bản)
import cv2

print("Bắt đầu Ingestion...")
video_paths = glob.glob('/content/data/**/videos/**/*.mp4', recursive=True)

points = []
for path in video_paths[:100]:  # Demo limit
    video_id = os.path.basename(path).split('.')[0]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): continue
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    middle_frame = total_frames // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)
    ret, frame = cap.read()
    if ret:
        frame_ms = int((middle_frame / fps) * 1000)
        tmp_img = "/tmp/tmp.jpg"
        cv2.imwrite(tmp_img, frame)
        img_vec = encode_image(tmp_img)
        text_vec = encode_text("") # Không có caption gốc
        
        if img_vec:
            points.append(PointStruct(
                id=str(uuid.uuid4()),
                vector={"image_siglip": img_vec, "text_e5": text_vec},
                payload={"video_id": video_id, "frame_ms": frame_ms}
            ))
    cap.release()

if points:
    client.upsert(collection_name="aic_collection", points=points)
print(f"Đã nạp {len(points)} keyframes vào Qdrant.")

In [ ]:
# 3. TRUY VẤN & FORMAT OUTPUT (TRAKE Logic)
import json
print("Bắt đầu xử lý Tasks...")

task_files = glob.glob('/content/tasks/*.jsonl')
submission = {"predictions": []}

for t_file in task_files:
    with open(t_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines:
            task_data = json.loads(line)
            task_id = task_data.get("task_id", "UNKNOWN")
            query = task_data.get("query", "")
            
            # Encode truy vấn bằng E5 (Text)
            q_vec = encode_text(query)
            
            # Search trên Qdrant
            search_res = client.search(
                collection_name="aic_collection",
                query_vector=("text_e5", q_vec),
                limit=10
            )
            
            # RRF hoặc Lọc tuần tự TRAKE sẽ được áp dụng tại đây.
            # Định dạng thành Rank 1-10
            results = []
            for i, hit in enumerate(search_res):
                results.append({
                    "rank": i + 1,
                    "video_id": hit.payload["video_id"],
                    "frame_ms": hit.payload["frame_ms"]
                })
                
            submission["predictions"].append({
                "task_id": task_id,
                "results": results
            })

# Xuất file JSONL
output_file = "submission.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(json.dumps(submission, ensure_ascii=False) + "\n")
print(f"Hoàn tất! Kết quả được lưu tại {output_file}")